# ASCAD fixed-key: συνέχεια στο ίδιο CNN baseline

Ένα notebook, τέσσερις εκπαιδεύσεις συνολικά. Το πρώτο benchmark των 3 epochs έχει εκτελεστεί.
Το παρόν αρχείο εντοπίζει/επαναφέρει code, ASCAD και checkpoint από το ενιαίο resume Input.
Η προεπιλογή STAGE="baseline" συνεχίζει το υπάρχον none run έως 50 epochs.
Τα υπόλοιπα στάδια compare/attack παραμένουν στο ίδιο notebook, μετά τον validation έλεγχο.
Ένα seed δίνει διερευνητικό αποτέλεσμα. Η διορθωμένη πρώτη cell έχει ελεγχθεί τοπικά·
η διόρθωση δεν έχει ακόμη εκτελεστεί στο Kaggle. Ο κώδικας εκπαίδευσης διατηρεί την ίδια έκδοση.


In [ ]:
"""Discover project/data and restore existing runs before importing the training package."""
import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def project_sha256(root):
    root = Path(root)
    files = sorted([*root.glob("src/**/*.py"), *root.glob("scripts/*.py"),
                    *root.glob("configs/*.json"), root / "pyproject.toml"])
    digest = hashlib.sha256()
    for path in files:
        digest.update(path.relative_to(root).as_posix().encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def archive_project_sha256(path):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        required = {"pyproject.toml", "src/sca/train.py", "requirements.txt", "tests/test_core.py"}
        if not required.issubset(names):
            return None
        selected = sorted(name for name in names if name == "pyproject.toml" or
                          (name.startswith("src/") and name.endswith(".py")) or
                          (name.startswith("scripts/") and name.count("/") == 1 and name.endswith(".py")) or
                          (name.startswith("configs/") and name.count("/") == 1 and name.endswith(".json")))
        digest = hashlib.sha256()
        for name in selected:
            digest.update(name.encode())
            digest.update(archive.read(name))
        return digest.hexdigest()


def extract_checked(path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(path) as archive:
        if sum(member.file_size for member in archive.infolist()) > 256 * 1024 * 1024:
            raise ValueError("Unexpectedly large project/run input archive")
        for member in archive.infolist():
            relative = PurePosixPath(member.filename)
            if (relative.is_absolute() or ".." in relative.parts or
                    "\\" in member.filename or ":" in member.filename or
                    stat.S_ISLNK(member.external_attr >> 16) or
                    not (destination / member.filename).resolve().is_relative_to(destination)):
                raise ValueError("Unsafe archive member: " + member.filename)
        destination.mkdir(parents=True, exist_ok=True)
        archive.extractall(destination)


def bootstrap_inputs(input_root, project_root, run_root, expected_dataset_sha256,
                     expected_source_sha256=None, require_checkpoint=True):
    input_root, project_root, run_root = map(Path, (input_root, project_root, run_root))
    staging = project_root.parent / "input_assets"
    search_roots = [input_root]
    print("Input folders:", [str(p) for p in sorted(input_root.iterdir())] if input_root.exists() else [])
    containers = [p for p in input_root.rglob("*.zip")
                  if p.stem.startswith(("kaggle_resume_input", "results"))]
    for container in sorted(containers):
        extracted = staging / file_sha256(container)[:16]
        marker = extracted / ".bootstrap_complete"
        if not marker.exists():
            extract_checked(container, extracted)
            marker.write_text(file_sha256(container), encoding="utf-8")
        search_roots.append(extracted)
    code_candidates, dataset_candidates, restore_candidates, restore_folders = [], [], [], []
    for search_root in search_roots:
        for path in search_root.rglob("pyproject.toml"):
            candidate = path.parent
            if (candidate / "src/sca/train.py").is_file() and (candidate / "requirements.txt").is_file():
                code_candidates.append((candidate, project_sha256(candidate), False))
        for path in search_root.rglob("*.zip"):
            identity = archive_project_sha256(path)
            if identity is not None:
                code_candidates.append((path, identity, True))
        dataset_candidates.extend(search_root.rglob("ASCAD*.h5"))
        restore_candidates.extend(search_root.rglob("sca_runs_minimal_v1.zip"))
        restore_folders.extend(path.parent.parent for path in search_root.rglob("last.pt")
                               if path.parent.name == "none_seed0" and
                               (path.parent / "manifest.json").is_file() and
                               (path.parent / "history.json").is_file())
    if (project_root / "src/sca/train.py").is_file():
        code_candidates.append((project_root, project_sha256(project_root), False))
    print("Code sources:", [(str(p), h) for p, h, _ in code_candidates])
    print("ASCAD files:", [str(p) for p in dataset_candidates])
    print("Restore archives:", [str(p) for p in restore_candidates])
    print("Extracted run folders:", [str(p) for p in restore_folders])
    valid_datasets = [p for p in sorted(set(dataset_candidates))
                      if file_sha256(p) == expected_dataset_sha256]
    if not valid_datasets:
        raise RuntimeError("ASCAD.h5 is missing or has the wrong checksum. Add the original data Input or kaggle_resume_input.zip.")
    restores = {file_sha256(p): p for p in sorted(set(restore_candidates))}
    if len(restores) > 1 and not (run_root / "none_seed0/last.pt").exists():
        raise RuntimeError("Different restore archives are attached. Keep only the intended checkpoint Input.")
    restore = next(iter(restores.values()), None)
    folders = {file_sha256(p / "none_seed0/last.pt"): p for p in sorted(set(restore_folders))}
    checkpoint_hashes = set(folders)
    for archive_path in restores.values():
        with zipfile.ZipFile(archive_path) as archive:
            checkpoint_hashes.add(hashlib.sha256(archive.read("none_seed0/last.pt")).hexdigest())
    if len(checkpoint_hashes) > 1 and not (run_root / "none_seed0/last.pt").exists():
        raise RuntimeError("Different restore checkpoints are attached. Keep only the intended checkpoint Input.")
    restore_folder = next(iter(folders.values()), None)
    manifest_path = run_root / "none_seed0/manifest.json"
    if manifest_path.exists():
        checkpoint_source = json.loads(manifest_path.read_text())["code"]["source_sha256"]
    elif restore is not None:
        with zipfile.ZipFile(restore) as archive:
            checkpoint_source = json.loads(archive.read("none_seed0/manifest.json"))["code"]["source_sha256"]
    elif restore_folder is not None:
        checkpoint_source = json.loads((restore_folder / "none_seed0/manifest.json").read_text())["code"]["source_sha256"]
    else:
        checkpoint_source = expected_source_sha256
    if expected_source_sha256 and checkpoint_source != expected_source_sha256:
        raise RuntimeError("The checkpoint uses a different code version.")
    matches = [(p, h, is_archive) for p, h, is_archive in code_candidates
               if checkpoint_source is None or h == checkpoint_source]
    if not matches:
        raise RuntimeError("Project code is missing or does not match the checkpoint. Add kaggle_project.zip or kaggle_resume_input.zip.")
    if len({h for _, h, _ in matches}) > 1:
        raise RuntimeError("Different code versions are attached. Select the intended project Input.")
    candidate, identity, is_archive = matches[0]
    if project_root.exists():
        if not (project_root / "src/sca/train.py").is_file() or project_sha256(project_root) != identity:
            raise RuntimeError("Existing working project is incomplete/different. Use a fresh session with the resume Input.")
    elif is_archive:
        extract_checked(candidate, project_root)
    else:
        shutil.copytree(candidate, project_root)
    if project_sha256(project_root) != identity:
        raise RuntimeError("Extracted project fingerprint mismatch")
    last_checkpoint = run_root / "none_seed0/last.pt"
    if not last_checkpoint.exists() and (restore is not None or restore_folder is not None):
        if run_root.exists() and any(run_root.iterdir()):
            raise RuntimeError("Existing run folder is incomplete; refusing to overwrite it.")
        if restore is not None:
            extract_checked(restore, run_root)
        else:
            run_root.mkdir(parents=True, exist_ok=True)
            for name in ("none_seed0", "noise_seed0", "shift_seed0", "combined_seed0"):
                if (restore_folder / name).is_dir():
                    shutil.copytree(restore_folder / name, run_root / name)
            for name in ("study_summary.json", "protocol_freeze.json"):
                if (restore_folder / name).is_file():
                    shutil.copy2(restore_folder / name, run_root / name)
    if require_checkpoint and not last_checkpoint.exists():
        raise RuntimeError("Checkpoint Input is missing. Add sca_runs_minimal_v1.zip or kaggle_resume_input.zip to continue baseline.")
    if last_checkpoint.exists():
        restored = json.loads((run_root / "none_seed0/manifest.json").read_text())
        if restored["code"]["source_sha256"] != identity or restored["dataset_sha256"] != expected_dataset_sha256:
            raise RuntimeError("Restored run does not match the selected code/data")
        history = json.loads((run_root / "none_seed0/history.json").read_text())
        print("Existing checkpoint kept. Completed epoch:", history[-1]["epoch"])
    print("Project:", project_root, "Dataset:", valid_datasets[0], "Source SHA256:", identity)
    return valid_datasets[0]

import os
import sys
import subprocess
import time

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
PROJECT_ROOT = Path("/kaggle/working/hardware_sca")
INPUT_ROOT = Path("/kaggle/input")
RUN_ROOT = Path("/kaggle/working/runs/minimal_v1")
DATASET_PATH = bootstrap_inputs(
    INPUT_ROOT, PROJECT_ROOT, RUN_ROOT,
    expected_dataset_sha256="f56625977fb6db8075ab620b1f3ef49a2a349ae75511097505855376e9684f91",
    expected_source_sha256="e641690a163be60d7b7288f1ead8e9d1726a00e403a26af1146a41b1a38d8f87",
    require_checkpoint=True,
)
os.chdir(PROJECT_ROOT)


## Environment

Η εγκατάσταση απαιτεί Internet on. Αν έχεις offline pinned wheels ή επιλέξεις το Kaggle
preinstalled runtime, βάλε `PIN_DEPENDENCIES=False`. Οι πραγματικές εκδόσεις αποθηκεύονται.
Αν το kernel έχει ήδη εισαγάγει παλιές εκδόσεις, κάνε restart και ξανατρέξε τις cells.


In [ ]:
PIN_DEPENDENCIES = True
if PIN_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.8.0",
                    "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt",
                    "setuptools==78.1.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", "."], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import torch
from sca.train import train, environment, write_json, code_identity
from sca.data import inspect_dataset
from sca.evaluate import evaluate_cached

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU in Notebook Settings before the GPU experiment")
print(environment(torch.device("cuda")))
print(inspect_dataset(DATASET_PATH, verify_attack=False))
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(Path("/kaggle/working") / "environment-freeze.txt").write_text(freeze)


In [ ]:
# Enter the quota shown in your account; no default quota is assumed.
AVAILABLE_GPU_HOURS = None
SESSION_LIMIT_HOURS = None
STAGE = "baseline"  # benchmark, baseline, compare, attack
PROTOCOL_FROZEN = False
RUN_TAG = "minimal_v1"
RESTORE_ARCHIVE = None  # Previous sca_runs_minimal_v1.zip added as a private Input, if needed.

base = json.loads(Path("configs/baseline_cnn.json").read_text())
matrix = json.loads(Path("configs/matrix.json").read_text())
validation_config = json.loads(Path("configs/evaluation_final.json").read_text())
validation_config["conditions"] = [c for c in validation_config["conditions"]
    if c["name"] in ("clean", "combined_matched", matrix["primary_condition"])]
validation_config["repetitions"] = 20
benchmark_evaluation = json.loads(Path("configs/evaluation_pilot.json").read_text())
run_root = Path("/kaggle/working/runs") / RUN_TAG
if STAGE not in ("benchmark", "baseline", "compare", "attack"):
    raise ValueError("Unknown STAGE")
if matrix["training_budgets"] != [base["n_train"]] or matrix["seeds"] != [base["seed"]]:
    raise ValueError("This notebook supports the minimal one-budget, one-seed study")
if len(matrix["strategies"]) != matrix["training_runs"] or matrix["training_runs"] != 4:
    raise ValueError("The minimal study requires exactly four strategies")
if RESTORE_ARCHIVE and (not run_root.exists() or not any(run_root.iterdir())):
    run_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(RESTORE_ARCHIVE) as previous:
        for member in previous.infolist():
            if not (run_root / member.filename).resolve().is_relative_to(run_root.resolve()):
                raise ValueError("Unsafe run archive member")
        previous.extractall(run_root)
write_json(Path("/kaggle/working") / "account_budget.json", {
    "available_gpu_hours_user_entered": AVAILABLE_GPU_HOURS,
    "session_limit_hours_user_entered": SESSION_LIMIT_HOURS,
    "device": environment(torch.device("cuda")), "code": code_identity()})


In [ ]:
import math

def train_strategy(strategy, epochs):
    config = {**base, "dataset": str(DATASET_PATH), "device": "cuda", "epochs": epochs,
              "run_dir": str(run_root / (strategy + "_seed0")),
              "augmentation": matrix["strategies"][strategy]}
    resume = (Path(config["run_dir"]) / "last.pt").exists()
    started = time.perf_counter()
    summary = train(config, resume=resume, reuse_completed=True)
    print(strategy, summary)
    write_json(Path(config["run_dir"]) / ("call_" + STAGE + ".json"), {
        "stage": STAGE, "training_call_seconds": time.perf_counter() - started,
        "requested_epochs": epochs, "resume_or_reuse": resume})
    return config, summary

if STAGE != "attack":
    config, summary = train_strategy("none", matrix["benchmark_epochs"] if STAGE == "benchmark" else base["epochs"])
    history = json.loads((Path(config["run_dir"]) / "history.json").read_text())
    measured = history[1:matrix["benchmark_epochs"]]
    steps_per_epoch = math.ceil(base["n_train"] / base["batch_size"])
    per_step = sum(r["train_seconds"] for r in measured) / (len(measured) * steps_per_epoch)
    total_steps = matrix["training_runs"] * base["epochs"] * steps_per_epoch
    completed_steps = sum(json.loads(p.read_text())[-1]["steps"]
                          for p in run_root.glob("*_seed0/history.json"))
    report = {"planned_unique_trainings": 4, "benchmark_additional_trainings": 0,
        "planned_total_steps": total_steps, "completed_steps": completed_steps,
        "reference_seconds_per_step_none": per_step,
        "estimated_remaining_training_hours_with_50pct_margin":
            max(0, total_steps - completed_steps) * per_step * 1.5 / 3600,
        "scope": "same GPU, none reference only; augmentation overhead unmeasured",
        "excluded_costs": "validation, evaluation, HPO, setup and I/O"}
    write_json(Path("/kaggle/working") / "gpu_cost_estimate.json", report)
    print(json.dumps(report, indent=2))
    diagnostics = benchmark_evaluation if STAGE == "benchmark" else validation_config
    evaluate_cached(config["run_dir"], diagnostics, device="cuda", split="validation",
                    dataset_override=DATASET_PATH)


Διάβασε το `gpu_cost_estimate.json` και τα validation αποτελέσματα πριν την πλήρη εκπαίδευση.
Τα benchmark runs δεν είναι ανεξάρτητα seed sweeps ούτε τελική ερευνητική απόδειξη.
Στο `baseline` συνεχίζονται οι πρώτες 3 epochs μέχρι 50· δεν ξεκινά νέο μοντέλο.
Έλεγξε το baseline στο validation πριν επιλέξεις `compare` για τις άλλες 3 στρατηγικές.
Ολοκληρωμένα runs και ίδιες αξιολογήσεις επαναχρησιμοποιούνται. Πάγωσε το πρωτόκολλο πριν το `attack`.


In [ ]:
if STAGE == "compare":
    for strategy in matrix["strategies"]:
        if strategy == "none":
            continue
        config, summary = train_strategy(strategy, base["epochs"])
        evaluate_cached(config["run_dir"], validation_config, device="cuda", split="validation",
                        dataset_override=DATASET_PATH)

if STAGE == "attack":
    if not PROTOCOL_FROZEN:
        raise RuntimeError("Freeze and review the protocol before final attack evaluation")
    final_config = json.loads(Path("configs/evaluation_final.json").read_text())
    records = {}
    current_source = code_identity()["source_sha256"]
    for strategy in matrix["strategies"]:
        path = run_root / (strategy + "_seed0")
        history = json.loads((path / "history.json").read_text())
        if history[-1]["epoch"] != base["epochs"]:
            raise RuntimeError("Complete all four trainings before the final attack stage")
        records[strategy] = json.loads((path / "manifest.json").read_text())
        actual = records[strategy]
        expected = {**base, "augmentation": matrix["strategies"][strategy], "device": "cuda"}
        ignored = {"dataset", "run_dir"}
        if {k: v for k, v in actual["config"].items() if k not in ignored} != {
            k: v for k, v in expected.items() if k not in ignored}:
            raise RuntimeError("Training config differs from the planned four-strategy comparison")
        if actual["code"]["source_sha256"] != current_source:
            raise RuntimeError("Code changed since training; review before freezing the protocol")
    freeze = {"matrix": matrix, "base_config": base, "evaluation_config": final_config,
              "training_manifests": records, "evaluation_code": code_identity()}
    freeze_path = run_root / "protocol_freeze.json"
    if freeze_path.exists() and json.loads(freeze_path.read_text()) != freeze:
        raise RuntimeError("Frozen protocol changed; do not tune after viewing attack results")
    write_json(freeze_path, freeze)
    for strategy in matrix["strategies"]:
        evaluate_cached(run_root / (strategy + "_seed0"), final_config, device="cuda", split="attack",
                        dataset_override=DATASET_PATH)


In [ ]:
# Save and download this archive plus the small output JSON files before ending the session.
if run_root.exists():
    summaries = {strategy: json.loads((run_root / (strategy + "_seed0") / "summary.json").read_text())
                 for strategy in matrix["strategies"]
                 if (run_root / (strategy + "_seed0") / "summary.json").exists()}
    write_json(run_root / "study_summary.json", {"stage": STAGE, "training_summaries": summaries,
        "unique_trainings": len(summaries), "planned_unique_trainings": 4,
        "completed_training_steps": sum(s["optimization_steps"] for s in summaries.values()),
        "measured_training_loop_seconds": sum(s["train_seconds"] for s in summaries.values()),
        "uncertainty": "one seed; no estimate of between-training variability"})
    archive_path = shutil.make_archive(str(Path("/kaggle/working") / ("sca_runs_" + RUN_TAG)), "zip", run_root)
    print("Download:", archive_path)
print("Notebook creation alone is not execution; retain Kaggle Save & Run All output")
